# Introduction to Python and Geopandas

## BIO597 Spatial Analysis of Biodiversity Fall 2026

## Goals for today
This notebook is intended to introduce you to the basic workflow of using Python within Jupyter notebooks,
using geopandas and species occurrence data as the motivating examples.

Here are some basic expectations for labs/assignments:
* I do **not** expect you to be python/programming experts coming into this class
* We will introduce and reinforce programming concepts gradually

### What are Jupyter Notebooks and why would I want to use one?
Jupyter notebooks are a great way to generate reproducible scientific analysis workflows in python. You can mix documentation, code, tables, and figures in one executable document which can easily be shared and published to the web. It's very common to organize/share notebooks in github. Here is a good example of a notebook demonstrating a method I am developing called [Multispecies EstimAtion of Dispersal On netWorks (Meadow)](https://github.com/isaacovercast/meadow/blob/main/notebooks/Community-Meadow.ipynb).

Let's get started.


#### This is your first markdown cell

Try running this cell by pressing the 'play' button in the toolbar above. What happens?

Markdown cells contain formatted text instead of Python code. They are useful for notes, explanations, headings, links, and instructions.

In [1]:
# This is your first actual code cell.

# python uses the pound sign ('#') to indicate a comment line (lines that will be ignored when running the code).

# Imports
# It's very standard to have the first code cell in a notebook import any needed packages

# Here we import geopandas, a simple python geospatial analysis package. 
# Saying 'as gpd' creates a synonym, so we can call
# functions from this package using the shorter name (saves typing).
import geopandas as gpd
# We also import the standard pandas python library for manipulating tabular data
import pandas as pd

# gpd and pd are nicknames (synonyms) for the full library names


## Species occurrence records

Data for this exercise is from [Harrington et al 2024 "Pleistocene Glaciation Drove Shared Population Coexpansion in Eastern North American Snakes"](https://onlinelibrary.wiley.com/doi/10.1111/mec.17625)

The `EasternSnakes` folder contains simple CSV files of occurrence records for several snake species. Each row is one record, with an `ID`, `Longitude`, and `Latitude`.

A CSV is just a table, so we will first read it with `pandas`, then convert it to a `GeoDataFrame` by turning the longitude and latitude columns into point geometries.


In [2]:
sdekayi_csv = "EasternSnakes/Sdekayi_coords.csv"
sdekayi_df = gpd.read_file(sdekayi_csv)
sdekayi_df.head()

,ID,Longitude,Latitude
0,CAS162020,-79.53386,33.14123
1,CAS199805,-82.80616589,41.54882969
2,CAS210409,-79.24691667,41.18311667
3,OMNH2693,-94.52158,34.69291
4,OMNH12346,-97.449726,35.17868


### Convert the table to a GeoDataFrame

We will start with DeKay's brownsnake (*Storeria dekayi*). The important step is `points_from_xy`, which creates point geometries from the longitude and latitude columns.

The coordinate reference system, or CRS, is `EPSG:4326` because these coordinates are longitude and latitude in decimal degrees. We will talk more about CRS later in the exercise and throughout the course.


In [3]:
sdekayi_gdf = gpd.GeoDataFrame(
    sdekayi_df,
    geometry=gpd.points_from_xy(sdekayi_df["Longitude"], sdekayi_df["Latitude"]),
    crs="EPSG:4326",
)

sdekayi_gdf["Species"] = "Storeria dekayi"

sdekayi_gdf.head()


,ID,Longitude,Latitude,geometry,Species
0,CAS162020,-79.53386,33.14123,POINT (-79.53386 33.14123),Storeria dekayi
1,CAS199805,-82.80616589,41.54882969,POINT (-82.80617 41.54883),Storeria dekayi
2,CAS210409,-79.24691667,41.18311667,POINT (-79.24692 41.18312),Storeria dekayi
3,OMNH2693,-94.52158,34.69291,POINT (-94.52158 34.69291),Storeria dekayi
4,OMNH12346,-97.449726,35.17868,POINT (-97.44973 35.17868),Storeria dekayi


### Plot the first species

geopandas geodataframes have a very useful function called `explore()` to plot the geometries on an interactive map. At this stage, we are mainly checking that the coordinates were read correctly and appear in a reasonable part of eastern North America.


In [4]:
sdekayi_gdf.explore()

### Load and plot a second species

Now repeat the same workflow for eastern copperhead (*Agkistrodon contortrix*). Repeating the steps once by hand is useful practice before wrapping the workflow in a function.


In [5]:
acontortrix_csv = "EasternSnakes/Acontortrix_coords.csv"
acontortrix_df = gpd.read_file(acontortrix_csv)

acontortrix_gdf = gpd.GeoDataFrame(
    acontortrix_df,
    geometry=gpd.points_from_xy(acontortrix_df["Longitude"], acontortrix_df["Latitude"]),
    crs="EPSG:4326",
)
acontortrix_gdf["Species"] = "Agkistrodon contortrix"

acontortrix_gdf.head()


,ID,Longitude,Latitude,geometry,Species
0,CAS203553,-90.49508231,38.96836261,POINT (-90.49508 38.96836),Agkistrodon contortrix
1,CAS203555,-90.91443611,37.92668333,POINT (-90.91444 37.92668),Agkistrodon contortrix
2,CAS203566,-91.52694444,35.39361111,POINT (-91.52694 35.39361),Agkistrodon contortrix
3,CAS203564,-90.46245,37.12165556,POINT (-90.46245 37.12166),Agkistrodon contortrix
4,KU337018,-96.60919,39.10677,POINT (-96.60919 39.10677),Agkistrodon contortrix


In [6]:
acontortrix_gdf.explore()

### Merge the two GeoDataFrames

Because both species tables have the same columns and the same CRS, we can stack them into one combined `GeoDataFrame` with `pd.concat`. Keeping a `Species` column lets us separate the records again later.


In [7]:
snakes_gdf = pd.concat([sdekayi_gdf, acontortrix_gdf])
snakes_gdf.explore()

### Color points by labels in a column

The previous figure shows the sampling distribution of both species combined, but it uses the same color for all points, which isn't very useful. We can tell `gdf.explore()` which column we are interested in for differentiating points by passing in the species ID column.

In [8]:
snakes_gdf.explore(column="Species")

### Choosing a color scheme

geopandas picks a default color scheme which sometimes works, and sometimes not. Here it didn't work so well because the light blue of *Storeria dekayi* is somewhat difficult to see. You can set your own colormap with the `cmap` parameter. Let's choose `rainbow` which gives nice results for 2 species.

You can see all the available colormaps in the [python colormap reference](https://matplotlib.org/stable/gallery/color/colormap_reference.html)

In [9]:
snakes_gdf.explore(column="Species", cmap="rainbow")

## A few basic spatial analyses

Once the records are in one `GeoDataFrame`, we can ask simple spatial questions. The examples below are intentionally basic: count records, draw a simple hull around the occurrence records for each species, estimate the total geographic area of each species.

For area e calculations, we first project the data from longitude/latitude into `EPSG:5070`, a projected CRS for North America with units in meters. Areas and distances in decimal degrees are hard to interpret, so this projection step matters.


In [10]:
# Count occurrence records for each species.
record_counts = snakes_gdf.groupby("Species").size()
record_counts


Species
Agkistrodon contortrix    32
Storeria dekayi           27
dtype: int64

In [13]:
# Project the combined records before measuring areas.
snakes_projected = snakes_gdf.to_crs("EPSG:5070")
snakes_projected.crs

<Projected CRS: EPSG:5070>
Name: NAD83 / Conus Albers
Axis Info [cartesian]:
- X[east]: Easting (metre)
- Y[north]: Northing (metre)
Area of Use:
- name: United States (USA) - CONUS onshore - Alabama; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming.
- bounds: (-124.79, 24.41, -66.91, 49.38)
Coordinate Operation:
- name: Conus Albers
- method: Albers Equal Area
Datum: North American Datum 1983
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

In [12]:
# Draw a simple convex hull around the records for each species.
# A convex hull is the smallest convex polygon that contains all points.
hulls_projected = snakes_projected.dissolve(by="Species").reset_index()

hulls_projected["geometry"] = hulls_projected.geometry.convex_hull

hulls = hulls_projected.to_crs(snakes_gdf.crs)

hulls.explore(column="Species", cmap="rainbow")


### Calculate the total area of the convex hull for each species

Geopandas polygons have an `area` property that is measured in square meters, so we can convert this to square kilometers by dividing by 1,000,000

In [ ]:
hulls["convex_hull_area_km2"] = hulls_projected.area / 1000000
hulls

### Practice submitting your work to your github

Now that you have run through this exercise, practice submitting your notebook code to your class repo.

Open a terminal window and run these commands to add, commit, and push your notebook:

```
# Go to the labs directory in the course repo
cd ~/BIO597-SpatialBiodiversity/docs/labs

# Add your changed lab
git add Lab01-IntroPython-Geopandas.ipynb
git commit -m 'Finished Lab01'
git push